In [1]:
import os
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv


In [ ]:

load_dotenv()

def get_top_tracks(limit=20):
    auth_manager = SpotifyOAuth(
        client_id=os.getenv('SPOTIPY_CLIENT_ID'),
        client_secret=os.getenv('SPOTIPY_CLIENT_SECRET'),
        redirect_uri=os.getenv('SPOTIPY_REDIRECT_URI'),
        scope="user-top-read"
    )
    sp = spotipy.Spotify(auth_manager=auth_manager)

    results = sp.current_user_top_tracks(
        limit=limit, 
        offset=0, 
        time_range='medium_term'
    )

    track_data = []
    for item in results['items']:
        track_info = {
            'id': item['id'],
            'name': item['name'],
            'artist': item['artists'][0]['name']
        }
        track_data.append(track_info)
    
    return track_data

if __name__ == "__main__":
    top_20 = get_top_tracks(20)
    print("Your Top 20 Tracks:")
    for i, track in enumerate(top_20, 1):
        print(f"{i}. {track['name']} by {track['artist']}")

Your Top 20 Tracks:
1. LOVE YOU FOR LIFE. by Loud Luxury
2. smoke by Łaszewo
3. Suburbs by Good Neighbours
4. Porch Light by Noah Kahan
5. Blackout by Breathe Carolina
6. Man I Need by Olivia Dean
7. The View Between Villages by Noah Kahan
8. the time by jigitz
9. Daisies by Good Neighbours
10. Baby Steps by Olivia Dean
11. Nice To Each Other by Olivia Dean
12. Colors - Ian Asher Remix by Halsey
13. Heartbeat by Jai Wolf
14. Backseat by Balu Brigada
15. Otherside by Poolside
16. CRASHING by Lost Kings
17. OUT LATE. by Loud Luxury
18. WHAT’S THE MOVE by Lost Kings
19. So Easy (To Fall In Love) by Olivia Dean
20. Mind Over Matter (Reprise) by Young the Giant


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import lancedb
import os
from tqdm import tqdm

def migrate_to_vector_db():
    MAIN_DB_PATH = r"D:\annas_archive_spotify_2025_07_metadata\spotify_clean.sqlite3"
    FEATURES_DB_PATH = r"D:\annas_archive_spotify_2025_07_metadata\spotify_clean_audio_features.sqlite3"
    VECTOR_DB_PATH = r"D:\annas_archive_spotify_2025_07_metadata\spotify_vector_vault"

    conn = sqlite3.connect(MAIN_DB_PATH)
    cursor = conn.cursor()
    
    conn.execute("PRAGMA journal_mode = OFF")
    conn.execute("PRAGMA synchronous = OFF")
    
    cursor.execute(f"ATTACH DATABASE '{FEATURES_DB_PATH}' AS features_db")

    print("Calculating global min/max values...")
    cursor.execute("SELECT MIN(popularity), MAX(popularity) FROM tracks")
    pop_min, pop_max = cursor.fetchone()

    cursor.execute("""
        SELECT 
            MIN(time_signature), MAX(time_signature),
            MIN(tempo), MAX(tempo),
            MIN(key), MAX(key),
            MIN(loudness), MAX(loudness)
        FROM features_db.track_audio_features
    """)
    ts_min, ts_max, tempo_min, tempo_max, key_min, key_max, loud_min, loud_max = cursor.fetchone()

    globals = {
        'popularity': (pop_min, pop_max),
        'time_signature': (ts_min, ts_max),
        'tempo': (tempo_min, tempo_max),
        'key': (key_min, key_max),
        'loudness': (loud_min, loud_max)
    }

    print("Estimating total unique tracks...")
    # total_tracks = cursor.execute("SELECT COUNT(*) FROM tracks WHERE popularity > 0").fetchone()[0]
    total_tracks = 45000000

    db = lancedb.connect(VECTOR_DB_PATH)
    
    cols_to_scale = ['time_signature', 'tempo', 'key', 'popularity', 'loudness']
    cols_already_scaled = [
        'speechiness', 'acousticness', 'instrumentalness', 
        'liveness', 'valence', 'danceability', 'energy'
    ]

    query = """
        SELECT 
            t.id, t.name, t.popularity,
            a.name as artist_name,
            f.time_signature, f.tempo, f.key, f.loudness,
            f.speechiness, f.acousticness, f.instrumentalness, 
            f.liveness, f.valence, f.danceability, f.energy
        FROM tracks t
        JOIN track_artists ta ON t.rowid = ta.track_rowid
        JOIN artists a ON ta.artist_rowid = a.rowid
        JOIN features_db.track_audio_features f ON t.id = f.track_id
        WHERE t.popularity > 0
    """



    chunk_size = 1000000
    first_chunk = True
    table = None

    print(f"Starting chunked migration of {total_tracks:,} unique tracks...")

    with tqdm(total=total_tracks, desc="Migrating", unit="track") as pbar:
        for chunk in pd.read_sql_query(query, conn, chunksize=chunk_size):
            for col in cols_to_scale:
                g_min, g_max = globals[col]
                if g_max > g_min:
                    chunk[col] = (chunk[col] - g_min) / (g_max - g_min)
                else:
                    chunk[col] = 0.0
            
            all_feature_cols = cols_to_scale + cols_already_scaled
            chunk['vector'] = chunk[all_feature_cols].values.tolist()
            
            clean_chunk = chunk[['id', 'name', 'artist_name', 'vector', 'popularity']]
            
            if first_chunk:
                table = db.create_table("tracks", data=clean_chunk, mode="overwrite")
                first_chunk = False
            else:
                table.add(clean_chunk)
            
            pbar.update(len(chunk))

    conn.close()
    print(f"Success! Cleaned vector database created at: {VECTOR_DB_PATH}")

if __name__ == "__main__":
    migrate_to_vector_db()

Calculating global min/max values...
Estimating total unique tracks...
Starting chunked migration of 45,000,000 unique tracks...


Migrating: 64282670track [52:40, 20341.15track/s]                           

Success! Cleaned vector database created at: D:\annas_archive_spotify_2025_07_metadata\spotify_vector_vault


: 

In [ ]:
import lancedb
import sqlite3
import pandas as pd
import numpy as np
import json
import os

# Paths
VECTOR_DB_PATH = r"D:\annas_archive_spotify_2025_07_metadata\spotify_vector_vault"
BOUNDS_FILE = r"D:\annas_archive_spotify_2025_07_metadata\spotify_bounds.json"
MAIN_DB = r"D:\annas_archive_spotify_2025_07_metadata\spotify_clean.sqlite3"
FEATURES_DB = r"D:\annas_archive_spotify_2025_07_metadata\spotify_clean_audio_features.sqlite3"

def get_recommendations(target_track_ids, limit=20, popularity_threshold=0.8):
    with open(BOUNDS_FILE, 'r') as f:
        b = json.load(f)

    conn = sqlite3.connect(MAIN_DB)
    conn.execute(f"ATTACH DATABASE '{FEATURES_DB}' AS f_db")
    
    placeholders = ','.join(['?'] * len(target_track_ids))
    query = f"""
        SELECT 
            t.popularity, f.time_signature, f.tempo, f.key, f.loudness,
            f.speechiness, f.acousticness, f.instrumentalness, 
            f.liveness, f.valence, f.danceability, f.energy
        FROM tracks t
        JOIN f_db.track_audio_features f ON t.id = f.track_id
        WHERE t.id IN ({placeholders})
    """
    
    user_tracks_df = pd.read_sql_query(query, conn, params=target_track_ids)
    conn.close()

    if user_tracks_df.empty:
        print("No matching tracks found in the database. Check your IDs.")
        return None

    cols_to_scale = ['time_signature', 'tempo', 'key', 'popularity', 'loudness']
    for col in cols_to_scale:
        v_min, v_max = b[col]['min'], b[col]['max']
        user_tracks_df[col] = (user_tracks_df[col] - v_min) / (v_max - v_min)

    feature_order = [
        'time_signature', 'tempo', 'key', 'popularity', 'loudness',
        'speechiness', 'acousticness', 'instrumentalness', 
        'liveness', 'valence', 'danceability', 'energy'
    ]
    
    profile_vector = user_tracks_df[feature_order].mean().values.astype(np.float32)

    db = lancedb.connect(VECTOR_DB_PATH)
    table = db.open_table("tracks")

    excluded_ids = ", ".join([f"'{tid}'" for tid in target_track_ids])
    where_clause = f"popularity > {popularity_threshold} AND id NOT IN ({excluded_ids})"

    results = (
        table.search(profile_vector)
        .where(where_clause)
        .limit(limit * 3)
        .to_pandas()
    )

    if results.empty:
        print("No matches found. Try lowering the popularity_threshold.")
        return None

    final_df = (
        results.groupby('id')
        .agg({
            'name': 'first',
            'artist_name': lambda x: ' & '.join(sorted(x.unique())),
            'popularity': 'first',
            '_distance': 'first'
        })
        .reset_index()
        .sort_values('_distance')
        .head(limit)
    )

    return final_df


if __name__ == "__main__":
    try:
        my_top_ids = [item['id'] for item in top_20]
        
        print(f"Finding the vibe for {len(my_top_ids)} tracks...")
        recommendations = get_recommendations(my_top_ids, limit=20, popularity_threshold=0.8)
        
        if recommendations is not None:
            print("\n--- YOUR TOP RECOMMENDATIONS ---")
            display_df = recommendations[['name', 'artist_name', 'popularity', '_distance']].copy()
            display_df['popularity'] = (display_df['popularity'] * 100).astype(int)
            print(display_df.to_string(index=False))
            
    except NameError:
        print("Error: The 'top_20' list is not defined in this scope.")

Finding the vibe for 20 tracks...

--- YOUR TOP RECOMMENDATIONS ---
                                 name                    artist_name  popularity  _distance
                                 Juna                         Clairo          82   0.414346
                               Medusa               Cameron Whitcomb          81   0.429070
                              Silence            Khalid & Marshmello          81   0.430799
     Vois sur ton chemin - Techno Mix                        BENNETT          81   0.436045
                               Tattoo                         Loreen          82   0.440256
                            Anti-Hero                   Taylor Swift          83   0.448816
                           Todo De Ti                 Rauw Alejandro          81   0.450134
                      Purple lace bra                     Tate McRae          81   0.451622
                      Angels Like You                    Miley Cyrus          81   0.454541
            